# Silver → Gold pipeline

Builds the gold analytics layer from the cleaned, deduplicated silver tables.

| Cell | Output | What it does |
| --- | --- | --- |
| Setup | — | imports and table names |
| Read silver | — | load `silver.crm_accounts`, `silver.organizations`, `silver.signins` |
| dim_accounts | `gold.dim_accounts` | SCD Type 1 upsert keyed on `account_id` |
| dim_organizations | `gold.dim_organizations` | SCD Type 2 license history; later runs merge the latest snapshot per org and skip the write when nothing changed |
| stg_org_attributes | `gold.stg_org_attributes` | latest non-versioned org attributes (tenant name, crm ref, test flag) |
| fact_signins_daily | `gold.fact_signins_daily` | daily signin aggregates per org |
| vw_mau | `gold.vw_mau` | monthly active users by account, plan and month (a view, not a table) |

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# --- table names (silver = cleaned input, gold = analytics output) ---
catalog = "coreview"

accounts_silver = f"{catalog}.silver.crm_accounts"
accounts_gold = f"{catalog}.gold.dim_accounts"

organizations_silver = f"{catalog}.silver.organizations"
organizations_gold = f"{catalog}.gold.dim_organizations"

signins_silver = f"{catalog}.silver.signins"
signins_gold = f"{catalog}.gold.fact_signins"  # only used by the disabled incremental reader below

### Incremental reading (not in use)

The cell below holds an earlier "read only what is new" helper, now commented out.
It is disabled because the silver tables are rebuilt from scratch on every run,
so there is nothing to filter incrementally — the MERGE and SCD2 logic already
detect the real changes. Kept for reference in case the pipeline is ever
switched to true incremental loads.

In [0]:
# --- read_incremental helper (DISABLED - kept for reference) ---
# This earlier approach read only the rows that were new since the last load:
# if the gold table already exists, keep just the silver rows whose
# Ingestion_timestamp is newer than the max already landed in gold.
# It is switched off because the silver tables are fully rebuilt on every
# run, so there is nothing to filter - the MERGE (dim_accounts) and the SCD2
# merge (dim_organizations) already detect and apply only the real changes.
# Re-enable it only if the pipeline is changed to true incremental loads.
# (The stale output shown below is from its last live run.)

# def read_incremental(silver_df, gold_table, timestamp_col="Ingestion_timestamp"):
#     """Read only the latest records: if the silver table already exists,
#     keep only bronze rows whose ingestion timestamp is newer than the max
#     already in silver. If silver does not exist yet, read the full table."""
#     if spark.catalog.tableExists(gold_table):
#         max_ts = spark.table(gold_table).agg(F.max(timestamp_col)).collect()[0][0]
#         if max_ts is not None:
#             print(f"incremental load from bronze -> {gold_table}: only rows after {max_ts}")
#             return silver_df.where(F.col(timestamp_col).cast("timestamp") > F.lit(max_ts))
#     print(f"full load (silver table {gold_table} not found or empty)")
#     return silver_df


# df_accounts = read_incremental(spark.table(accounts_silver), accounts_gold)
# df_organizations = read_incremental(spark.table(organizations_silver), organizations_gold)
# df_signins = read_incremental(spark.table(signins_silver), signins_gold)

# print(f"crm_accounts:    {df_accounts.count()} rows")
# print(f"organizations:   {df_organizations.count()} rows")
# print(f"signins:        {df_signins.count()} rows")

In [0]:
# --- read the silver inputs ---
df_accounts = spark.table(accounts_silver)
df_organizations = spark.table(organizations_silver)
df_signins = spark.table(signins_silver)

print(f"crm_accounts:  {df_accounts.count()} rows")
print(f"organizations: {df_organizations.count()} rows")
print(f"signins:       {df_signins.count()} rows")

In [0]:
# --- dim_accounts: SCD Type 1 upsert (latest row wins, no history kept) ---
key = "account_id"  # business key for the merge

# stamp the load time so every run is traceable in the gold table
df_accounts = (
    df_accounts
    .withColumn("Ingestion_timestamp", F.current_timestamp())
    .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

df_accounts.createOrReplaceTempView("accounts_updates")

if spark.catalog.tableExists(accounts_gold):
    # upsert: existing accounts are updated in place, new ones inserted
    spark.sql(f"""
    MERGE INTO {accounts_gold} AS tgt
    USING accounts_updates AS src
      ON tgt.`{key}` = src.`{key}`
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
else:
    # target does not exist yet: create it with all rows from the source
    spark.sql(f"""
    CREATE TABLE {accounts_gold}
    AS SELECT * FROM accounts_updates
    """)

print(f"dim_accounts now has {spark.table(accounts_gold).count()} rows")

In [0]:

# --- dim_organizations: Batch-based SCD Type 2 license history ---

# 1. Prepare source data
df = df_organizations.withColumn(
    "snapshot_date",
    F.col("capture_date").cast("date")
)
window_by_date = Window.partitionBy("org_id").orderBy("snapshot_date")

df = (
    df
    .withColumn("row_num", F.row_number().over(window_by_date))
    .withColumn("prev_plan", F.lag("license_plan").over(window_by_date))
    .withColumn("prev_seats", F.lag("license_seats").over(window_by_date))
    .withColumn("prev_status", F.lag("status").over(window_by_date))
)

changed = (
    (F.col("row_num") == 1)
    | ~F.col("license_plan").eqNullSafe(F.col("prev_plan"))
    | ~F.col("license_seats").eqNullSafe(F.col("prev_seats"))
    | ~F.col("status").eqNullSafe(F.col("prev_status")))
df = df.withColumn("change_flag", F.when(changed, 1).otherwise(0))
df = df.withColumn("version", F.sum("change_flag").over(window_by_date.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
versions = df.groupBy("org_id", "version").agg(
    F.min("snapshot_date").alias("valid_from"),
    F.first("license_plan").alias("plan"),
    F.first("license_seats").alias("seats"),
    F.first("status").alias("status")
)
version_window = Window.partitionBy("org_id").orderBy("version")

dim_organizations = (
    versions
    .withColumn("valid_to", F.lead("valid_from").over(version_window))
    .withColumn("is_current", F.col("valid_to").isNull())
    .drop("version"))
dim_organizations.write.mode("overwrite").saveAsTable(
    organizations_gold
)
org_attributes = (
    df_organizations
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("org_id").orderBy(F.desc("capture_date"))
        )
    )
    .filter(F.col("rn") == 1)
    .select(
        "org_id",
        "tenant_name",
        "crm_ref",
        "is_test"
    )
)

org_attributes.write.mode("overwrite").saveAsTable(
    f"{catalog}.gold.stg_org_attributes"
)
print(f"Rows in {organizations_gold}: {spark.table(organizations_gold).count()}")
print(f"Current records: {spark.table(organizations_gold).filter(F.col('is_current')).count()}")


In [0]:
# --- fact_signins_daily: daily signin aggregates per org ---
# NOTE: distinct_users here is per org per DAY - do not sum it into monthly
# numbers (that double-counts users across days); use vw_mau for MAU.

# 1. clean rows: non-null org_id, user_id, result and a valid event date
df_clean = (
    df_signins
    .select(
        "org_id",
        "user_id",
        F.col("result"),
        F.col("event_date")
    )
    .filter(
        F.col("org_id").isNotNull()
        & F.col("user_id").isNotNull()
        & F.col("result").isNotNull()
        & F.col("event_date").isNotNull()
    )
)

# 2. keep only signins from organizations we know about (present in dim_organizations)
known_orgs = spark.table(organizations_gold).select("org_id").distinct()
df_clean = df_clean.join(known_orgs, "org_id", "left_semi")

# 3. daily aggregates per org
fact_signins_daily = (
    df_clean.groupBy("org_id", "event_date")
    .agg(
        F.sum(F.when(F.col("result") == "success", 1).otherwise(0)).alias("successful_signins"),
        F.sum(F.when(F.col("result") == "failure", 1).otherwise(0)).alias("failed_signins"),
        F.countDistinct(F.when(F.col("result") == "success", F.col("user_id"))).alias("distinct_users"),
    )
)

fact_signins_daily.write.mode("overwrite").saveAsTable(f"{catalog}.gold.fact_signins_daily")

print(f"fact_signins_daily: {spark.table(f'{catalog}.gold.fact_signins_daily').count()} rows")


In [0]:
# --- Monthly Active Users (MAU) by account, plan and month ---
# Built from the deduplicated sign-in events (silver), NOT from the daily
# fact table: summing per-day/per-org distinct users would double-count,
# so MAU is a single DISTINCT user_id count over the whole month.
# Exposed as a view: the pipeline lives in the view SQL itself, because a
# persistent view cannot reference a temporary (session-scoped) view.

# 1. deduplicated events: successful sign-ins only, month from event_ts
# 2. each org's plan for that month, from the SCD2 dim:
#    the version where valid_from <= month_end < coalesce(valid_to, 9999-12-31)
# 3. org -> account link: crm_ref's first 15 chars are the Salesforce key
#    (18-char refs are the same key plus a 3-char checksum suffix);
#    test tenants and orgs with no account link (null crm_ref) are dropped
# 4. MAU: distinct users per account, month and plan
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vw_mau AS
WITH events AS (
    SELECT
        org_id,
        user_id,
        date_format(date_trunc('MM', event_ts), 'yyyy-MM') AS month,
        last_day(CAST(date_trunc('MM', event_ts) AS DATE)) AS month_end
    FROM {signins_silver}
    WHERE result = 'success'
),
events_plan AS (
    SELECT e.org_id, e.user_id, e.month, d.plan
    FROM events e
    JOIN {organizations_gold} d
      ON  d.org_id = e.org_id
      AND d.valid_from <= e.month_end
      AND e.month_end < COALESCE(d.valid_to, DATE '9999-12-31')
),
linked AS (
    SELECT a.account_id, ep.user_id, ep.month, ep.plan
    FROM events_plan ep
    JOIN {catalog}.gold.stg_org_attributes oa
      ON oa.org_id = ep.org_id
    JOIN {accounts_gold} a
      ON SUBSTRING(oa.crm_ref, 1, 15) = a.crm_key
    WHERE NOT oa.is_test
)
SELECT account_id, month, plan, COUNT(DISTINCT user_id) AS mau
FROM linked
GROUP BY account_id, month, plan
""")

print(f"vw_mau: {spark.table(f'{catalog}.gold.vw_mau').count()} rows")

In [0]:
# --- row counts for the gold tables and view ---
print(f"dim_accounts:      {spark.table(f'coreview.gold.dim_accounts').count()} rows")
print(f"dim_organizations: {spark.table(f'coreview.gold.dim_organizations').count()} rows")
print(f"stg_org_attributes:{spark.table(f'coreview.gold.stg_org_attributes').count()} rows")
print(f"fact_signins_daily:{spark.table(f'coreview.gold.fact_signins_daily').count()} rows")
print(f"vw_mau:            {spark.table(f'coreview.gold.vw_mau').count()} rows")